# 04 - Modelos de Regresión: Capa Premium

**Objetivo**: entrenar los modelos que alimentan la **capa premium** de GoalPredict 2026.

**Targets a predecir**:

| # | Target | Fuente | Volumen | Tipo modelo |
|---|---|---|---|---|
| 1 | Goles totales esperados | Martj42 | 30.929 | Regresión |
| 2 | Over/Under 2.5 goles | Martj42 | 30.929 | Clasificación binaria |
| 3 | Goles 1er tiempo / 2do tiempo | Fjelstul (1T/2T) | ~1.800 muestras | Regresión |
| 4 | Tarjetas amarillas por equipo | Fjelstul (bookings) | ~1.800 | Regresión |
| 5 | Tarjetas rojas por equipo | Fjelstul (bookings) | ~1.800 | Clasif. binaria |
| 6 | Corners / Posesión / Faltas por equipo | WC2022 | 64 partidos × 2 = 128 | Baseline (promedio histórico por equipo) |

**Nota sobre los WC2022 features**: 128 muestras es muy poco para regresión robusta. Usaremos un modelo baseline tipo "promedio histórico del equipo + tendencia global". Cuando llegue el Mundial 2026 con más datos, podremos reentrenar con regresión completa.

**Métricas**: MAE, RMSE, R² para regresión; accuracy/F1/log-loss para clasificación binaria.

**Persistencia**: cada modelo se guarda como `.pkl` en `ml/trained_models/` con metadata.


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]  # ml/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, log_loss, roc_auc_score,
)
from sklearn.dummy import DummyRegressor, DummyClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

from src.data import (
    load_matches as load_fjelstul_matches,
    load_goals as load_fjelstul_goals,
    load_bookings as load_fjelstul_bookings,
    load_wc2022_matches,
)
from src.features import get_feature_columns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
RANDOM_STATE = 42

# Carpeta para los modelos entrenados
MODELS_DIR = ROOT / 'trained_models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Goles totales esperados + Over/Under 2.5 (Martj42)

Usamos el dataset Martj42 ya procesado en `02_feature_engineering.ipynb`. Las mismas features que el clasificador W/D/L, pero el target ahora es `total_goals = home_score + away_score`.


In [ ]:
df = pd.read_csv(ROOT / 'data/processed/match_features.csv', parse_dates=['date'])
df['total_goals'] = df['home_score'] + df['away_score']
df['over_2_5'] = (df['total_goals'] > 2.5).astype(int)

FEATURE_COLS = get_feature_columns()
CAT_COLS = ['tournament_group']
ALL_FEATURES = FEATURE_COLS + CAT_COLS

train = df[df['date'] < '2018-01-01']
test = df[df['date'] >= '2018-01-01']
X_train, X_test = train[ALL_FEATURES], test[ALL_FEATURES]
y_train_goals, y_test_goals = train['total_goals'], test['total_goals']
y_train_over, y_test_over = train['over_2_5'], test['over_2_5']

print(f'Train: {len(train):,} | Test: {len(test):,}')
print(f'Goles totales — train: μ={y_train_goals.mean():.2f}, std={y_train_goals.std():.2f}')
print(f'P(Over 2.5) — train: {y_train_over.mean():.2%}')


In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), FEATURE_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_COLS),
])


### 1.1 Regresión de goles totales

In [ ]:
# Baseline: predecir la media histórica
dummy_reg = Pipeline([('pre', preprocessor), ('reg', DummyRegressor(strategy='mean'))])
dummy_reg.fit(X_train, y_train_goals)
yp = dummy_reg.predict(X_test)
mae_dummy = mean_absolute_error(y_test_goals, yp)
rmse_dummy = np.sqrt(mean_squared_error(y_test_goals, yp))
print(f'Baseline (media) — MAE: {mae_dummy:.3f}, RMSE: {rmse_dummy:.3f}, R²: 0.000')

# Random Forest Regressor
rf_goals = Pipeline([('pre', preprocessor), ('reg', RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=15, n_jobs=-1, random_state=RANDOM_STATE,
))])
%time rf_goals.fit(X_train, y_train_goals)
yp_rf = rf_goals.predict(X_test)
print(f'RF — MAE: {mean_absolute_error(y_test_goals, yp_rf):.3f}, '
      f'RMSE: {np.sqrt(mean_squared_error(y_test_goals, yp_rf)):.3f}, '
      f'R²: {r2_score(y_test_goals, yp_rf):.3f}')

# Gradient Boosting / XGBoost Regressor
if HAS_XGB:
    xgb_goals = Pipeline([('pre', preprocessor), ('reg', XGBRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        n_jobs=-1, random_state=RANDOM_STATE,
    ))])
    %time xgb_goals.fit(X_train, y_train_goals)
    yp_xgb = xgb_goals.predict(X_test)
    print(f'XGB — MAE: {mean_absolute_error(y_test_goals, yp_xgb):.3f}, '
          f'RMSE: {np.sqrt(mean_squared_error(y_test_goals, yp_xgb)):.3f}, '
          f'R²: {r2_score(y_test_goals, yp_xgb):.3f}')
    best_goals_model = xgb_goals
else:
    best_goals_model = rf_goals


In [ ]:
# Persistir el modelo de goles
joblib.dump({
    'model': best_goals_model,
    'feature_cols': FEATURE_COLS, 'cat_cols': CAT_COLS,
    'target': 'total_goals',
    'training_metrics': {
        'mae': float(mean_absolute_error(y_test_goals, best_goals_model.predict(X_test))),
        'rmse': float(np.sqrt(mean_squared_error(y_test_goals, best_goals_model.predict(X_test)))),
        'r2': float(r2_score(y_test_goals, best_goals_model.predict(X_test))),
    },
}, MODELS_DIR / 'regressor_total_goals.pkl')
print(f'✓ regressor_total_goals.pkl guardado')


### 1.2 Clasificador Over/Under 2.5 goles

In [ ]:
if HAS_XGB:
    over_clf = Pipeline([('pre', preprocessor), ('clf', XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', n_jobs=-1, random_state=RANDOM_STATE,
    ))])
else:
    over_clf = Pipeline([('pre', preprocessor), ('clf', LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE,
    ))])

%time over_clf.fit(X_train, y_train_over)
yp = over_clf.predict(X_test)
yp_p = over_clf.predict_proba(X_test)[:, 1]

print(f'Over/Under 2.5 — accuracy: {accuracy_score(y_test_over, yp):.3f}, '
      f'F1: {f1_score(y_test_over, yp):.3f}, '
      f'AUC: {roc_auc_score(y_test_over, yp_p):.3f}, '
      f'log-loss: {log_loss(y_test_over, yp_p):.3f}')

joblib.dump({
    'model': over_clf,
    'feature_cols': FEATURE_COLS, 'cat_cols': CAT_COLS,
    'target': 'over_2_5',
    'training_metrics': {
        'accuracy': float(accuracy_score(y_test_over, yp)),
        'f1': float(f1_score(y_test_over, yp)),
        'auc': float(roc_auc_score(y_test_over, yp_p)),
        'log_loss': float(log_loss(y_test_over, yp_p)),
    },
}, MODELS_DIR / 'classifier_over_2_5.pkl')
print('✓ classifier_over_2_5.pkl guardado')


## 2. Goles por tiempo y tarjetas (Fjelstul)

Fjelstul tiene **goles con `match_period`** (primer tiempo, segundo tiempo, extra time) y **tarjetas con flags** `yellow_card`/`red_card`/`second_yellow_card`. Esto nos permite predecir lo que Martj42 no tiene desagregado.

**Limitación**: solo 900 partidos de Mundial × 2 perspectivas (equipo local + visitante) = 1.800 muestras. Modelos más simples para evitar overfitting.


In [ ]:
# Construir un dataset por equipo-partido desde Fjelstul
fjelstul_matches = load_fjelstul_matches()
fjelstul_goals = load_fjelstul_goals()
fjelstul_bookings = load_fjelstul_bookings()

print(f'Fjelstul matches: {len(fjelstul_matches):,}')
print(f'Fjelstul goals: {len(fjelstul_goals):,}')
print(f'Fjelstul bookings: {len(fjelstul_bookings):,}')

# Goles por (match_id, team_id, periodo)
goals = fjelstul_goals.copy()
goals['period'] = goals['match_period'].fillna('').apply(
    lambda s: 'first_half' if 'first half' in s
    else 'second_half' if 'second half' in s
    else 'extra' if 'extra' in s else 'other'
)
goals_per_team = goals.groupby(['match_id', 'player_team_id', 'period']).size().unstack(fill_value=0)
for p in ['first_half', 'second_half']:
    if p not in goals_per_team.columns:
        goals_per_team[p] = 0
goals_per_team = goals_per_team[['first_half', 'second_half']].reset_index()
goals_per_team.columns = ['match_id', 'team_id', 'goals_1t', 'goals_2t']

# Tarjetas por (match_id, team_id)
cards_per_team = fjelstul_bookings.groupby(['match_id', 'team_id']).agg(
    yellow=('yellow_card', 'sum'),
    red=('red_card', 'sum'),
    second_yellow=('second_yellow_card', 'sum'),
).reset_index()

# Construir tabla equipo-partido a partir de matches (1 fila por equipo)
from src.data import load_team_appearances
team_apps = load_team_appearances()

# Merge
fj_long = team_apps.merge(goals_per_team, on=['match_id', 'team_id'], how='left')
fj_long['goals_1t'] = fj_long['goals_1t'].fillna(0).astype(int)
fj_long['goals_2t'] = fj_long['goals_2t'].fillna(0).astype(int)
fj_long = fj_long.merge(cards_per_team, on=['match_id', 'team_id'], how='left')
fj_long[['yellow', 'red', 'second_yellow']] = fj_long[['yellow', 'red', 'second_yellow']].fillna(0).astype(int)

print(f'\nfj_long: {fj_long.shape}')
fj_long[['team_name','opponent_name','goals_for','goals_against','goals_1t','goals_2t','yellow','red']].head()



### 2.1 Features para los modelos Fjelstul

Construimos rolling stats por equipo dentro de Fjelstul. Como son pocos partidos por equipo, usamos ventana móvil = 5.


In [ ]:
# Asegurar ordenación cronológica
fj_long = fj_long.sort_values(['team_id', 'match_date']).reset_index(drop=True)

# Rolling features: promedio de los últimos 5 partidos en Mundial
ROLL_WIN = 5
g = fj_long.groupby('team_id', group_keys=False)
for col_in, col_out in [
    ('goals_for', 'roll5_gf'),
    ('goals_against', 'roll5_ga'),
    ('yellow', 'roll5_yellow'),
    ('red', 'roll5_red'),
    ('goals_1t', 'roll5_g1t'),
    ('goals_2t', 'roll5_g2t'),
]:
    fj_long[col_out] = g[col_in].apply(lambda s: s.shift(1).rolling(ROLL_WIN, min_periods=1).mean()).reset_index(level=0, drop=True)

# Mismas features para el OPONENTE — hacemos un self-join por (match_id, team_id) ↔ (match_id, opponent_id)
opp_features = fj_long[['match_id', 'team_id', 'roll5_gf', 'roll5_ga', 'roll5_yellow', 'roll5_red']].rename(
    columns={'team_id': 'opponent_id',
             'roll5_gf': 'opp_roll5_gf',
             'roll5_ga': 'opp_roll5_ga',
             'roll5_yellow': 'opp_roll5_yellow',
             'roll5_red': 'opp_roll5_red'}
)
fj_long = fj_long.merge(opp_features, on=['match_id', 'opponent_id'], how='left')

# Flag de localía y stage del torneo
fj_long['is_home'] = (fj_long['home_team'] == 1).astype(int)
fj_long['is_knockout'] = (fj_long['knockout_stage'] == 1).astype(int)

FJ_FEATURES = [
    'roll5_gf', 'roll5_ga', 'roll5_yellow', 'roll5_red', 'roll5_g1t', 'roll5_g2t',
    'opp_roll5_gf', 'opp_roll5_ga', 'opp_roll5_yellow', 'opp_roll5_red',
    'is_home', 'is_knockout',
]

# Filtrar filas con poco historial (rolling NaN)
ready = fj_long.dropna(subset=FJ_FEATURES).reset_index(drop=True)
print(f'Muestras listas para entrenamiento: {len(ready):,}')
print(f'Features: {FJ_FEATURES}')


### 2.2 Goles 1T / 2T (con selección automática best-vs-baseline)

**Estrategia honesta**: probamos 4 candidatos (DummyRegressor mean, Ridge, RF shallow, GB shallow) y nos quedamos con el de **menor MAE** en test. Si ningún modelo supera el DummyRegressor (predecir la media histórica), persistimos el dummy — predicciones constantes pero no engañosas.

Esto es importante porque con solo ~1.400 muestras de entrenamiento, los modelos complejos pueden tener R² negativo (peor que el promedio). En esos casos el promedio histórico es la mejor predicción posible y debe ser lo que se persiste.


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge

# Split temporal: usar Mundial 2014 y 2018 como test
test_tournaments = ['WC-2014', 'WC-2018']
tr = ready[~ready['tournament_id'].isin(test_tournaments)]
te = ready[ready['tournament_id'].isin(test_tournaments)]
print(f'Train: {len(tr)} muestras | Test: {len(te)} muestras ({test_tournaments})')

fj_pre = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])


def pick_best_regressor(target_name, ytr_arr, yte_arr):
    """Prueba candidatos y devuelve el de menor MAE en test.

    Si ningún modelo supera el DummyRegressor (predice media histórica),
    devuelve el dummy — evita persistir modelos con R² negativo.
    """
    candidates = {
        'dummy_mean': DummyRegressor(strategy='mean'),
        'ridge': Pipeline([('pre', fj_pre), ('reg', Ridge(alpha=10.0, random_state=RANDOM_STATE))]),
        'rf_shallow': Pipeline([('pre', fj_pre), ('reg', RandomForestRegressor(
            n_estimators=200, max_depth=3, min_samples_leaf=30, n_jobs=-1, random_state=RANDOM_STATE))]),
        'gb_shallow': Pipeline([('pre', fj_pre), ('reg', GradientBoostingRegressor(
            n_estimators=100, max_depth=2, learning_rate=0.03, min_samples_leaf=30, random_state=RANDOM_STATE))]),
    }
    results = []
    for name, model in candidates.items():
        if name == 'dummy_mean':
            model.fit(np.zeros((len(ytr_arr), 1)), ytr_arr)
            yp = model.predict(np.zeros((len(yte_arr), 1)))
        else:
            model.fit(tr[FJ_FEATURES], ytr_arr)
            yp = model.predict(te[FJ_FEATURES])
        mae = mean_absolute_error(yte_arr, yp)
        rmse = np.sqrt(mean_squared_error(yte_arr, yp))
        r2 = r2_score(yte_arr, yp)
        results.append((name, model, mae, rmse, r2))

    print(f'\n[{target_name}]')
    for n, _, mae, rmse, r2 in results:
        print(f'  {n:14s} MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}')

    best = min(results, key=lambda x: x[2])
    print(f'  → Elegido: {best[0]}')
    return best[1], best[0], {'mae': float(best[2]), 'rmse': float(best[3]), 'r2': float(best[4])}


model_g1t, alg_g1t, m_g1t = pick_best_regressor('goles_1t', tr['goals_1t'].values, te['goals_1t'].values)
model_g2t, alg_g2t, m_g2t = pick_best_regressor('goles_2t', tr['goals_2t'].values, te['goals_2t'].values)

joblib.dump({'model': model_g1t, 'features': FJ_FEATURES, 'target': 'goals_1t',
             'algorithm': alg_g1t, 'training_metrics': m_g1t},
            MODELS_DIR / 'regressor_goals_1t.pkl')
joblib.dump({'model': model_g2t, 'features': FJ_FEATURES, 'target': 'goals_2t',
             'algorithm': alg_g2t, 'training_metrics': m_g2t},
            MODELS_DIR / 'regressor_goals_2t.pkl')
print(f'\n✓ regressor_goals_1t.pkl ({alg_g1t}) y regressor_goals_2t.pkl ({alg_g2t}) guardados')


### 2.3 Tarjetas amarillas (best-vs-baseline) y rojas (clasificación binaria)

Misma lógica para amarillas. Rojas como clasificación binaria con LogReg balanceado.


In [ ]:
# Amarillas: regresión con selección automática
model_yellow, alg_yellow, m_yellow = pick_best_regressor(
    'yellow', tr['yellow'].values, te['yellow'].values,
)

# Rojas: clasificación binaria (¿hay roja sí/no?) — son raras (~5% partidos)
print(f'\n[red_card binary]')
y_red_tr = (tr['red'] > 0).astype(int)
y_red_te = (te['red'] > 0).astype(int)
print(f'  P(roja) train: {y_red_tr.mean():.2%}, test: {y_red_te.mean():.2%}')

model_red = Pipeline([('pre', fj_pre), ('clf', LogisticRegression(
    max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE,
))])
model_red.fit(tr[FJ_FEATURES], y_red_tr)
yp_red = model_red.predict(te[FJ_FEATURES])
yp_red_p = model_red.predict_proba(te[FJ_FEATURES])[:, 1]
m_red = {
    'accuracy': float(accuracy_score(y_red_te, yp_red)),
    'f1': float(f1_score(y_red_te, yp_red)) if y_red_te.sum() > 0 else None,
    'auc': float(roc_auc_score(y_red_te, yp_red_p)) if y_red_te.nunique() > 1 else None,
}
print(f'  acc={m_red["accuracy"]:.3f}  F1={m_red["f1"]}  AUC={m_red["auc"]}')

joblib.dump({'model': model_yellow, 'features': FJ_FEATURES, 'target': 'yellow',
             'algorithm': alg_yellow, 'training_metrics': m_yellow},
            MODELS_DIR / 'regressor_yellow_cards.pkl')
joblib.dump({'model': model_red, 'features': FJ_FEATURES, 'target': 'has_red_card',
             'training_metrics': m_red},
            MODELS_DIR / 'classifier_red_card.pkl')
print(f'\n✓ regressor_yellow_cards.pkl ({alg_yellow}) y classifier_red_card.pkl guardados')


## 3. Corners, Posesión y Faltas (WC2022)

Solo tenemos 64 partidos × 2 perspectivas = 128 muestras. No es suficiente para un Random Forest robusto. Usamos un **baseline de promedios por equipo**: predecimos el valor esperado como el promedio histórico del equipo en el dataset (con un fallback al promedio global cuando el equipo no tiene historia).

**Implementación**: una clase `TeamAveragePredictor` que aprende `{team_name: avg_value}` del entrenamiento y predice usando ese diccionario.

Cuando el Mundial 2026 termine, podremos reentrenar con más datos y subir a Ridge o Random Forest.


In [ ]:
class TeamAveragePredictor:
    """Baseline: predice el promedio histórico de cada equipo (fallback: media global)."""
    def __init__(self):
        self.team_means_ = {}
        self.global_mean_ = None
    def fit(self, team_names, y):
        df = pd.DataFrame({'team': team_names, 'y': y})
        self.team_means_ = df.groupby('team')['y'].mean().to_dict()
        self.global_mean_ = float(np.mean(y))
        return self
    def predict(self, team_names):
        return np.array([self.team_means_.get(t, self.global_mean_) for t in team_names])

wc = load_wc2022_matches()

# Convertir a formato long (1 fila por equipo)
rows = []
for _, r in wc.iterrows():
    for side, team_col, opp_col in [(1, 'team1', 'team2'), (2, 'team2', 'team1')]:
        rows.append({
            'team': r[team_col],
            'opponent': r[opp_col],
            'possession': r[f'possession_team{side}'],
            'corners': r[f'corners_team{side}'],
            'fouls': r[f'fouls_against_team{side}'],
            'attempts': r[f'total_attempts_team{side}'],
            'on_target': r[f'on_target_attempts_team{side}'],
            'yellow_wc': r[f'yellow_cards_team{side}'],
        })
wc_long = pd.DataFrame(rows)
print(f'WC2022 long: {wc_long.shape}')
print(wc_long.describe().round(2))


In [ ]:
# Entrenar baselines por equipo para cada target premium
wc_models = {}
for target in ['possession', 'corners', 'fouls', 'attempts', 'on_target']:
    valid = wc_long.dropna(subset=[target])
    pred = TeamAveragePredictor().fit(valid['team'].values, valid[target].values)
    wc_models[target] = pred
    # Métricas en train (no hay test split por escasez de datos)
    yp = pred.predict(valid['team'].values)
    mae = mean_absolute_error(valid[target], yp)
    print(f'  {target:12s} — μ={valid[target].mean():.2f}, equipos únicos: {valid[target].count()}, MAE in-sample: {mae:.2f}')

joblib.dump({
    'models': wc_models,
    'targets': list(wc_models.keys()),
    'source': 'wc2022',
    'note': 'Baseline por promedio histórico del equipo en WC2022. Solo 64 partidos disponibles.',
}, MODELS_DIR / 'baseline_wc2022_stats.pkl')
print('\n✓ baseline_wc2022_stats.pkl guardado')


In [ ]:
# Visualización de las distribuciones (informativo para la UI)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, target in zip(axes, ['possession', 'corners', 'fouls']):
    wc_long[target].plot(kind='hist', bins=15, ax=ax, color='#2E86AB', alpha=0.8)
    ax.axvline(wc_long[target].mean(), color='red', linestyle='--', label=f'μ={wc_long[target].mean():.1f}')
    ax.set_title(f'{target.title()} por equipo (WC2022)')
    ax.legend()
plt.tight_layout(); plt.show()


## 4. Resumen de modelos premium entrenados

Archivos generados en `ml/trained_models/`:

| Archivo | Target | Tipo | Fuente | Métrica clave |
|---|---|---|---|---|
| `regressor_total_goals.pkl` | Goles totales del partido | Regresión | Martj42 (30k) | MAE / R² |
| `classifier_over_2_5.pkl` | Over/Under 2.5 goles | Clasif. binaria | Martj42 (30k) | AUC / log-loss |
| `regressor_goals_1t.pkl` | Goles del equipo en 1er tiempo | Regresión | Fjelstul (~1.8k) | MAE |
| `regressor_goals_2t.pkl` | Goles del equipo en 2do tiempo | Regresión | Fjelstul (~1.8k) | MAE |
| `regressor_yellow_cards.pkl` | Tarjetas amarillas del equipo | Regresión | Fjelstul (~1.8k) | MAE |
| `classifier_red_card.pkl` | ¿Hay tarjeta roja? | Clasif. binaria | Fjelstul (~1.8k) | AUC |
| `baseline_wc2022_stats.pkl` | Posesión, corners, faltas, intentos por equipo | Baseline (promedio histórico) | WC2022 (64) | MAE in-sample |

**Sobre la calidad esperada**:
- Los modelos de **goles totales y Over/Under** deberían dar R² razonable (~0.1-0.2) porque Martj42 da mucho volumen.
- **Goles 1T/2T y tarjetas amarillas** tienen pocas muestras (Fjelstul ~1.8k) — esperar MAE comparable al baseline, mejora marginal.
- **Tarjetas rojas** son raras (~10% partidos) — el modelo solo es útil si mejora claramente el AUC > 0.6.
- **Corners/Posesión/Faltas** son baselines tipo "promedio del equipo" — no son ML real pero dan algo a la UI mientras llega más data.

## Próximos pasos

1. **`05_clustering.ipynb`**: K-Means para clusters de estilo (ofensivo, defensivo, equilibrado) sobre features agregadas por equipo. Útil como narrativa en la UI premium.
2. **Backend FastAPI**: cargar `classifier.pkl` + estos `.pkl` y exponer endpoints `/predict/result`, `/predict/goals`, `/predict/cards`, `/predict/stats`.
3. **Frontend React + Tailwind**: selector de equipos + visualización de probabilidades.
